In [ ]:
from pathlib import Path
import os, psutil

# repo & data
REPO_ROOT = (Path.cwd() / ".." / "..").resolve()
DATASET = REPO_ROOT / "data" / "raw" / "labeled" / "BGL" / "BGL.log"

# outputs
OUT = (REPO_ROOT / "data" / "results" / "bgl_realtime_streaming_nohash").resolve()
for d in ["windowed", "anomalies", "metrics"]:  # no wide event_matrix by default
    (OUT / d).mkdir(parents=True, exist_ok=True)

# streaming params
WINDOW_SIZE = "10m"
ALLOWED_LATENESS = "0s"
CHUNK_SIZE = 25_000

# template vocabulary (fixed after warmup)
TOP_K_TEMPLATES = 20000  # try 5k–10k; +1 "OTHER" bucket gets added automatically

# PCA / detection
WARMUP_WINDOWS = 5000  # windows to build vocabulary + frozen model
VARIANCE_THRESHOLD = 0.90  # choose k by cumulative variance
ALPHA_SPE = 1e-2  # tail probability for SPE threshold
ALPHA_T2 = 2e-3  # tail probability for T² threshold
USE_SCALING = True
USE_T2 = True  # alert when SPE>thr_SPE OR T²>thr_T2

# performance knobs
cpu_count = psutil.cpu_count(logical=False) or psutil.cpu_count()
os.environ["POLARS_MAX_THREADS"] = str(cpu_count)
os.environ["RAYON_NUM_THREADS"] = str(cpu_count)

print("Dataset exists:", DATASET.exists())
print("Output:", OUT)

In [ ]:
import polars as pl
import numpy as np
from collections import Counter, defaultdict
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from log_anomaly_analysis.core.config.loader import load_config
from log_anomaly_analysis.core.components.preprocessing import PreprocessorComponent
from log_anomaly_analysis.core.components.parsing import TemplateParserComponent

def parse_duration(spec: str) -> timedelta:
    if not spec:
        return timedelta(0)
    try:
        unit = spec[-1].lower(); val = float(spec[:-1])
    except Exception:
        return timedelta(seconds=float(spec))
    return {"s": timedelta(seconds=val),"m": timedelta(minutes=val),"h": timedelta(hours=val)}.get(unit, timedelta(seconds=val))

def select_k(ev_ratio: np.ndarray, thr: float) -> int:
    c = np.cumsum(ev_ratio)
    k = int(np.searchsorted(c, thr, side="left") + 1)
    return max(1, min(k, ev_ratio.shape[0]))

# numerically stable SPE (residual energy)
def spe_scores(Xs: np.ndarray, comps: np.ndarray, mean_vec: np.ndarray) -> np.ndarray:
    C = Xs - mean_vec           # (n,p)
    T = C @ comps.T             # (n,k)
    R = C - T @ comps           # (n,p)
    return np.einsum("ij,ij->i", R, R).astype(np.float32)

# Hotelling T² (optional)
def t2_scores(Xs: np.ndarray, comps: np.ndarray, mean_vec: np.ndarray, eigs: np.ndarray, k: int) -> np.ndarray:
    C = Xs - mean_vec
    T = C @ comps[:k].T
    lam = np.maximum(eigs[:k], 1e-12)
    return np.sum((T**2) / lam, axis=1).astype(np.float32)


In [ ]:
class AccNoHash:
    """
    Aggregates templates per tumbling window as Counters, plus labels & counts.
    """
    def __init__(self, win_size, lateness):
        self.wtd = parse_duration(win_size)
        self.ltd = parse_duration(lateness)
        self.cnts   = defaultdict(Counter)  # WindowStart -> Counter(template -> count)
        self.counts = defaultdict(int)      # WindowStart -> total log lines
        self.labels = defaultdict(bool)     # WindowStart -> any anomaly label
        self.max_ts = None

    def _key(self, ts: datetime):
        epoch = datetime(1970,1,1,tzinfo=ts.tzinfo)
        s = (ts-epoch).total_seconds(); size = self.wtd.total_seconds() or 1.0
        return epoch + timedelta(seconds=int(s//size)*size)

    def update_chunk(self, ts_list, tpl_list, label_list):
        for ts, tpl, lab in zip(ts_list, tpl_list, label_list):
            if ts is None: continue
            self.max_ts = ts if self.max_ts is None else max(self.max_ts, ts)
            k = self._key(ts)
            if tpl is not None:
                self.cnts[k][str(tpl)] += 1
            self.counts[k] += 1
            if lab is not None and lab != "-":
                self.labels[k] = True

    def _should_close(self, k):
        return self.max_ts and (k + self.wtd <= self.max_ts - self.ltd)

    def finalize_ready(self):
        ready = sorted([k for k in self.cnts if self._should_close(k)])
        wins, mats, ys = [], [], []
        for k in ready:
            c  = self.cnts.pop(k)
            lc = self.counts.pop(k, 0)
            y  = self.labels.pop(k, False)
            wins.append({"Window": k, "WindowStart": k, "WindowEnd": k+self.wtd, "LogCount": lc})
            mats.append((c, lc))  # keep as (Counter, total_logs)
            ys.append(y)
        return wins, mats, ys


In [ ]:
def iterate_file(path: Path, chunk_size: int):
    with path.open("r", encoding="utf-8", errors="ignore") as fh:
        chunk = []
        for ln in fh:
            s = ln.strip()
            if s: chunk.append(s)
            if len(chunk) >= chunk_size:
                yield chunk; chunk = []
        if chunk: yield chunk

# parquet part counters
part = {"windowed": 0, "anomalies": 0, "metrics": 0}
def write_df(dir_path: Path, name: str, df: pl.DataFrame):
    if df is None or df.is_empty(): return
    p = dir_path / f"part-{part[name]:05d}.parquet"
    df.write_parquet(p); part[name] += 1

# online confusion-matrix accumulator
cm = {"TP": 0, "FP": 0, "FN": 0, "TN": 0}
def update_metrics(y_true: np.ndarray, y_pred: np.ndarray, cm=cm):
    cm["TP"] += int(((y_pred == 1) & (y_true == 1)).sum())
    cm["FP"] += int(((y_pred == 1) & (y_true == 0)).sum())
    cm["FN"] += int(((y_pred == 0) & (y_true == 1)).sum())
    cm["TN"] += int(((y_pred == 0) & (y_true == 0)).sum())

def current_metrics(cm=cm):
    p = cm["TP"] / (cm["TP"] + cm["FP"]) if (cm["TP"] + cm["FP"]) > 0 else 0.0
    r = cm["TP"] / (cm["TP"] + cm["FN"]) if (cm["TP"] + cm["FN"]) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1

# fixed vocab (+ OTHER) vectorization
def build_vocab_from_warmup(warm_mats, top_k: int):
    df_counter = Counter()
    for c,_ in warm_mats:
        df_counter.update(set(c.keys()))    # document frequency across windows
    most = [t for t,_ in df_counter.most_common(top_k)]
    vocab = {t:i for i,t in enumerate(most)}
    other_index = len(vocab)                # last index for OTHER
    return vocab, other_index, other_index + 1

def vectorize_counter(c: Counter, log_count: int, vocab: dict, other_idx: int, p: int, freq_norm=True):
    x = np.zeros(p, dtype=np.float32)
    if c:
        for t, n in c.items():
            i = vocab.get(t, other_idx)
            x[i] += float(n)
    if freq_norm and log_count > 0:
        x /= float(log_count)
    return x


In [ ]:
cfg    = load_config(REPO_ROOT / "configs" / "labeled" / "BGL.yaml")
pre    = PreprocessorComponent(cfg.preprocessing.__dict__)
parser = TemplateParserComponent(cfg.parsing.__dict__)

acc = AccNoHash(WINDOW_SIZE, ALLOWED_LATENESS)

# model state (frozen after warmup)
scaler = StandardScaler() if USE_SCALING else None
pca    = None
ksel   = 0
thr_spe = float("inf")
thr_t2  = float("inf")
warmed  = False

# vocab state
vocab = None
OTHER = None
P_DIM = None  # = len(vocab)+1

# warmup buffers
warm_mats, warm_labels = [], []
total_wins = 0


In [ ]:
for lines in iterate_file(DATASET, CHUNK_SIZE):
    # preprocess → templates
    df = pl.DataFrame({"raw_log": lines})
    pre_df = pre.process(df)
    if pre_df.is_empty(): continue

    parsed_df = parser.process(pre_df)
    if parsed_df.is_empty(): continue

    # choose a stable template key: prefer text template
    if "EventTemplate" in parsed_df.columns:
        tpl = parsed_df["EventTemplate"].to_list()
    else:
        tpl = parsed_df["TemplateId"].to_list()

    ts     = parsed_df["Timestamp"].to_list()
    labels = pre_df["Label"].to_list() if "Label" in pre_df.columns else [None]*len(ts)

    # update window accumulator
    acc.update_chunk(ts, tpl, labels)
    wins, mats, ys = acc.finalize_ready()
    if not (wins or mats): continue

    # write window metadata (good for audit)
    win_df = pl.DataFrame(wins) if wins else pl.DataFrame()
    if not win_df.is_empty():
        write_df(OUT / "windowed", "windowed", win_df)

    y_true = np.array(ys, dtype=bool) if ys else np.array([], dtype=bool)
    total_wins += len(wins)

    # --- warmup / calibration: build vocab + batch PCA on NORMAL windows only ---
    if (not warmed) and mats:
        warm_mats.extend(mats)
        warm_labels.extend(y_true.tolist())
        if len(warm_mats) >= WARMUP_WINDOWS:
            # keep only NORMAL windows for calibration
            normal_pairs = [pair for pair, lab in zip(warm_mats, warm_labels) if not lab]
            if not normal_pairs:
                normal_pairs = warm_mats   # fallback: use all if labels unavailable

            # 1) build fixed vocab (top-K) + OTHER
            vocab, OTHER, P_DIM = build_vocab_from_warmup(normal_pairs, TOP_K_TEMPLATES)

            # 2) build dense X for calibration
            Xw = np.vstack([vectorize_counter(c, lc, vocab, OTHER, P_DIM, True)
                            for (c, lc) in normal_pairs]).astype(np.float32)

            # 3) fit scaler + PCA (batch)
            scaler = StandardScaler() if USE_SCALING else None
            if USE_SCALING:
                scaler.fit(Xw)
                Xw = scaler.transform(Xw).astype(np.float32)

            pca = PCA(svd_solver="full")
            pca.fit(Xw)

            ksel = select_k(pca.explained_variance_ratio_, VARIANCE_THRESHOLD)

            # 4) empirical thresholds on calibration windows
            S_warm = spe_scores(Xw, pca.components_[:ksel], pca.mean_)
            thr_spe = float(np.quantile(S_warm, 1.0 - ALPHA_SPE))

            if USE_T2:
                T2_warm = t2_scores(Xw, pca.components_, pca.mean_, pca.explained_variance_, ksel)
                thr_t2  = float(np.quantile(T2_warm, 1.0 - ALPHA_T2))

            warmed = True
            warm_mats.clear(); warm_labels.clear()

    # --- online scoring (frozen model) ---
    if warmed and mats:
        # vectorize current batch against fixed vocab
        X = np.vstack([vectorize_counter(c, lc, vocab, OTHER, P_DIM, True)
                       for (c, lc) in mats]).astype(np.float32)
        Xs = scaler.transform(X).astype(np.float32) if USE_SCALING else X

        S = spe_scores(Xs, pca.components_[:ksel], pca.mean_)
        if USE_T2:
            T2 = t2_scores(Xs, pca.components_, pca.mean_, pca.explained_variance_, ksel)
            yhat = (S > thr_spe) | (T2 > thr_t2)
        else:
            yhat = (S > thr_spe)

        update_metrics(y_true.astype(int), yhat.astype(int))

        # write anomalies for this flush
        anom_df = pl.DataFrame({
            "Window":      win_df["Window"] if "Window" in win_df.columns else [None]*len(S),
            "WindowStart": win_df["WindowStart"] if "WindowStart" in win_df.columns else [None]*len(S),
            "WindowEnd":   win_df["WindowEnd"] if "WindowEnd" in win_df.columns else [None]*len(S),
            "LogCount":    [lc for (_, lc) in mats],
            "AnomalyScore_SPE": S,
            **({"AnomalyScore_T2": T2} if USE_T2 else {}),
            "IsAnomaly":   yhat,
        })
        write_df(OUT / "anomalies", "anomalies", anom_df)

    # streaming metrics snapshot (append only after warmup)
    if warmed:
        p, r, f1 = current_metrics()
        met = pl.DataFrame({
            "WindowsProcessed": [total_wins],
            "Precision": [p], "Recall": [r], "F1": [f1],
            "k": [ksel], "SPE_threshold": [thr_spe],
            **({"T2_threshold":[thr_t2]} if USE_T2 else {}),
            "VocabSize": [P_DIM],
        })
        write_df(OUT / "metrics", "metrics", met)

print("Total windows processed:", total_wins)


In [ ]:
p, r, f1 = current_metrics()
out = {"Precision": p, "Recall": r, "F1": f1, "k": ksel, "SPE_threshold": thr_spe, "VocabSize": P_DIM}
if USE_T2: out["T2_threshold"] = thr_t2
print(out)
print("Outputs at:", OUT)


In [ ]:
import matplotlib.pyplot as plt

METRICS_DIR = OUT / "metrics"
FIG_DIR = OUT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(METRICS_DIR.glob("*.parquet"))
if not files:
    print(f"No metrics files found in {METRICS_DIR}")
else:
    # Load & tidy
    df = pl.concat([pl.read_parquet(f) for f in files], how="vertical", rechunk=True)
    if "WindowsProcessed" in df.columns:
        df = df.unique(subset=["WindowsProcessed"], keep="last").sort(
            "WindowsProcessed"
        )
    # Drop pre-warmup zeros
    df = df.filter(
        (pl.col("Precision") > 0) | (pl.col("Recall") > 0) | (pl.col("F1") > 0)
    )

    wp = df["WindowsProcessed"].to_numpy()
    prec = df["Precision"].to_numpy()
    rec = df["Recall"].to_numpy()
    f1 = df["F1"].to_numpy()

    k = df.get_column("k").to_numpy() if "k" in df.columns else None
    spe_thr = (
        df.get_column("SPE_threshold").to_numpy()
        if "SPE_threshold" in df.columns
        else None
    )
    t2_thr = (
        df.get_column("T2_threshold").to_numpy()
        if "T2_threshold" in df.columns
        else None
    )

    # Always returns SAME length as x (even window sizes too)
    def rolling_mean(x: np.ndarray, w: int = 100) -> np.ndarray:
        if x is None or x.size == 0:
            return x
        w = max(1, min(int(w), x.size))
        if w == 1:
            return x.astype(float)
        kernel = np.ones(w, dtype=float) / w
        left = (w - 1) // 2
        right = (w - 1) - left  # ensures left+right = w-1
        xp = np.pad(x, (left, right), mode="edge")
        y = np.convolve(xp, kernel, mode="valid")
        # y has length len(x); guaranteed
        return y.astype(float)

    SMOOTH = True
    WIN = 100  # any int; even now works

    prec_s = rolling_mean(prec, WIN) if SMOOTH else prec
    rec_s = rolling_mean(rec, WIN) if SMOOTH else rec
    f1_s = rolling_mean(f1, WIN) if SMOOTH else f1

    # --- Precision ---
    plt.figure(figsize=(10, 5))
    plt.plot(wp, prec, alpha=0.35, label="Precision (raw)")
    if SMOOTH:
        plt.plot(wp, prec_s, label=f"Precision (rolling mean {WIN})")
    plt.xlabel("Windows processed")
    plt.ylabel("Precision")
    plt.title("Precision over time")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "precision.png", dpi=144)
    plt.show()

    # --- Recall ---
    plt.figure(figsize=(10, 5))
    plt.plot(wp, rec, alpha=0.35, label="Recall (raw)")
    if SMOOTH:
        plt.plot(wp, rec_s, label=f"Recall (rolling mean {WIN})")
    plt.xlabel("Windows processed")
    plt.ylabel("Recall")
    plt.title("Recall over time")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "recall.png", dpi=144)
    plt.show()

    # --- F1 ---
    plt.figure(figsize=(10, 5))
    plt.plot(wp, f1, alpha=0.35, label="F1 (raw)")
    if SMOOTH:
        plt.plot(wp, f1_s, label=f"F1 (rolling mean {WIN})")
    plt.xlabel("Windows processed")
    plt.ylabel("F1 score")
    plt.title("F1 over time")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "f1.png", dpi=144)
    plt.show()

    print("Saved figures to:", FIG_DIR)